In [ ]:
!pip install -q PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.6 MB/s eta 0:00:00


In [4]:
import os
import tarfile
import fitz
import unicodedata
import pandas as pd

from google.colab import files, drive
from IPython.display import display, Image


In [5]:
# Choose input source:
# "drive" = use an archive already in Google Drive
# "computer" = upload an archive from your computer

input_source = "drive"

base_dir = os.path.join(
    "/content/drive/MyDrive",
    "Character Complexity"
)

# Input PDF archive - zipped file

if input_source == "drive":

    drive.mount("/content/drive")

    archive_path = "/content/drive/MyDrive/Character Complexity/udhr_pdf.tar.gz"

elif input_source == "computer":

    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file was uploaded.")

    archive_path = next(iter(uploaded))

else:

    raise ValueError(
        "input_source must be 'drive' or 'computer'."
    )


if not os.path.isfile(archive_path):

    raise FileNotFoundError(
        f"Input archive not found:\n{archive_path}"
    )


print("Input archive:")
print(archive_path)


# Output directories

pdf_output_dir = os.path.join(
    base_dir,
    "udhr_pdfs"
)

txt_output_dir = os.path.join(
    base_dir,
    "udhr_text_files"
)


os.makedirs(
    pdf_output_dir,
    exist_ok=True
)

os.makedirs(
    txt_output_dir,
    exist_ok=True
)


print()
print("PDF output:")
print(pdf_output_dir)

print()
print("Text output:")
print(txt_output_dir)


# Languages to extract

languages = {
    "English": "eng",
    "Portuguese": "por",
    "Japanese": "jpn",
    "Chinese_Mandarin": "chn",
    "Hindi": "hnd",
    "Spanish": "spn",
    "Arabic": "arz",
    "Russian": "rus",
    "Quechua_Ayacucho": "quy",
    "Hebrew": "hbr"
}


# Extract selected PDFs

extracted_files = []
missing_languages = []


with tarfile.open(
    archive_path,
    "r:gz"
) as tar:

    members = tar.getmembers()

    for language, lang_id in languages.items():

        matches = [
            member
            for member in members
            if os.path.basename(member.name)
            == f"{lang_id}.pdf"
        ]

        if not matches:

            print(
                f"Missing: {language} ({lang_id})"
            )

            missing_languages.append(language)

            continue


        member = matches[0]

        extracted_pdf_path = os.path.join(
            pdf_output_dir,
            f"{lang_id}.pdf"
        )


        extracted_file = tar.extractfile(
            member
        )

        if extracted_file is None:

            print(
                f"Could not extract: "
                f"{language} ({lang_id})"
            )

            missing_languages.append(language)

            continue


        with extracted_file as source:

            with open(
                extracted_pdf_path,
                "wb"
            ) as target:

                target.write(
                    source.read()
                )


        extracted_files.append(
            extracted_pdf_path
        )


        print(
            f"Extracted: {language} → "
            f"{extracted_pdf_path}"
        )


# Extraction summary

print()
print("PDF extraction complete.")

print(
    f"PDFs extracted: "
    f"{len(extracted_files):,}"
)

print(
    f"Languages missing/failed: "
    f"{len(missing_languages):,}"
)


if missing_languages:

    print()
    print("Missing or failed languages:")

    for language in missing_languages:
        print(f"  {language}")


print()
print("PDF directory:")
print(pdf_output_dir)

Mounted at /content/drive
Input archive:
/content/drive/MyDrive/Character Complexity/udhr_pdf.tar.gz

PDF output:
/content/drive/MyDrive/Character Complexity/udhr_pdfs

Text output:
/content/drive/MyDrive/Character Complexity/udhr_text_files
Extracted: English → /content/drive/MyDrive/Character Complexity/udhr_pdfs/eng.pdf
Extracted: Portuguese → /content/drive/MyDrive/Character Complexity/udhr_pdfs/por.pdf
Extracted: Japanese → /content/drive/MyDrive/Character Complexity/udhr_pdfs/jpn.pdf
Extracted: Chinese_Mandarin → /content/drive/MyDrive/Character Complexity/udhr_pdfs/chn.pdf
Extracted: Hindi → /content/drive/MyDrive/Character Complexity/udhr_pdfs/hnd.pdf
Extracted: Spanish → /content/drive/MyDrive/Character Complexity/udhr_pdfs/spn.pdf
Extracted: Arabic → /content/drive/MyDrive/Character Complexity/udhr_pdfs/arz.pdf
Extracted: Russian → /content/drive/MyDrive/Character Complexity/udhr_pdfs/rus.pdf
Extracted: Quechua_Ayacucho → /content/drive/MyDrive/Character Complexity/udhr_pdfs/

In [ ]:
# Functions for PDF inspection

def extract_pdf_text(pdf_path):
    """
    Extract text from a PDF and collect basic PDF statistics.
    """

    doc = fitz.open(pdf_path)

    page_texts = []
    total_images = 0
    pages_with_text = 0

    for page in doc:

        text = page.get_text()
        page_texts.append(text)

        if text.strip():
            pages_with_text += 1

        total_images += len(
            page.get_images(full=True)
        )

    doc.close()

    text = "".join(page_texts)

    return {
        "text": text,
        "pages": len(page_texts),
        "pages_with_text": pages_with_text,
        "images": total_images
    }


def inspect_text_quality(text):
    """
    Check for obvious PDF text-extraction problems.

    This checks technical Unicode/extraction problems.
    It does not determine whether the extracted language
    is linguistically correct.
    """

    if not text.strip():

        return {
            "quality": "NO TEXT",
            "reason": "No text extracted"
        }

    replacement_count = text.count("\ufffd")

    control_count = sum(
        unicodedata.category(char) == "Cc"
        and char not in "\n\r\t"
        for char in text
    )

    private_use_count = sum(
        unicodedata.category(char) == "Co"
        for char in text
    )

    if replacement_count > 0:

        return {
            "quality": "SUSPICIOUS",
            "reason": (
                f"Contains {replacement_count:,} "
                "Unicode replacement characters"
            )
        }

    if private_use_count > 0:

        return {
            "quality": "SUSPICIOUS",
            "reason": (
                f"Contains {private_use_count:,} "
                "private-use Unicode characters"
            )
        }

    if control_count > 0:

        return {
            "quality": "SUSPICIOUS",
            "reason": (
                f"Contains {control_count:,} "
                "unexpected control characters"
            )
        }

    return {
        "quality": "TEXT OK",
        "reason": (
            "No obvious Unicode extraction problems detected"
        )
    }


# Inspect all PDFs

pdf_results = []

# Keep successfully extracted text in memory.
# This avoids extracting the same PDF twice.

extracted_texts = {}


for filename in sorted(
    os.listdir(pdf_output_dir)
):

    if not filename.lower().endswith(".pdf"):
        continue

    pdf_path = os.path.join(
        pdf_output_dir,
        filename
    )

    try:

        result = extract_pdf_text(
            pdf_path
        )

        text = result["text"]

        quality = inspect_text_quality(
            text
        )

        # Store extracted text so it does not need
        # to be extracted from the PDF again later.

        extracted_texts[filename] = text

        pdf_results.append({
            "file": filename,
            "pages": result["pages"],
            "pages_with_text": result["pages_with_text"],
            "characters": len(text),
            "non_whitespace": len(text.strip()),
            "images": result["images"],
            "status": quality["quality"],
            "reason": quality["reason"]
        })

    except Exception as e:

        pdf_results.append({
            "file": filename,
            "pages": None,
            "pages_with_text": None,
            "characters": None,
            "non_whitespace": None,
            "images": None,
            "status": "ERROR",
            "reason": str(e)
        })


pdf_summary = pd.DataFrame(
    pdf_results
)


# Display PDF extraction report

print()
print("PDF extraction report")

display(
    pdf_summary
)


# Identify problematic PDFs

problem_statuses = {
    "NO TEXT",
    "SUSPICIOUS",
    "ERROR"
}

problems = pdf_summary[
    pdf_summary["status"].isin(
        problem_statuses
    )
]


print()
print("PDFs requiring inspection")

if not problems.empty:
    display(problems)
else:
    print("None")


# Display PDFs with apparently usable text

successful_files = pdf_summary[
    pdf_summary["status"] == "TEXT OK"
]["file"].tolist()


print()
print("PDFs with apparently usable text")

display(
    pdf_summary[
        pdf_summary["status"] == "TEXT OK"
    ]
)


print()
print(
    f"PDFs passing extraction check: "
    f"{len(successful_files):,}"
)

print(
    f"PDFs not automatically converted: "
    f"{len(pdf_summary) - len(successful_files):,}"
)


# Show text previews

print()
print("Text previews")

for filename in successful_files:

    text = extracted_texts[filename]

    print()
    print(filename)
    print(repr(text[:500]))


# Convert successful PDFs to UTF-8 text

conversion_results = []


for filename in successful_files:

    text = extracted_texts[filename]

    txt_filename = (
        os.path.splitext(filename)[0]
        + ".txt"
    )

    txt_path = os.path.join(
        txt_output_dir,
        txt_filename
    )

    try:

        # Use Unicode NFC as the canonical representation
        # for downstream character/glyph analysis.

        text = unicodedata.normalize(
            "NFC",
            text
        )

        # Verify that the normalized text can be
        # represented as UTF-8.

        text.encode("utf-8")

        is_nfc = unicodedata.is_normalized(
            "NFC",
            text
        )

        with open(
            txt_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(text)

        conversion_results.append({
            "file": filename,
            "txt_file": txt_filename,
            "characters": len(text),
            "NFC": is_nfc,
            "status": "SAVED"
        })

        print(
            f"{filename} → {txt_filename} "
            f"({len(text):,} characters)"
        )

    except Exception as e:

        conversion_results.append({
            "file": filename,
            "txt_file": txt_filename,
            "characters": None,
            "NFC": None,
            "status": f"ERROR: {e}"
        })

        print(
            f"{filename}: ERROR"
        )
        print(e)


conversion_table = pd.DataFrame(
    conversion_results
)


# Display conversion results

print()
print("Text conversion results")

display(
    conversion_table
)


# Verify resulting UTF-8 text files

print()
print("Verifying saved text files")

verification_results = []


for filename in sorted(
    os.listdir(txt_output_dir)
):

    if not filename.lower().endswith(".txt"):
        continue

    path = os.path.join(
        txt_output_dir,
        filename
    )

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

        verification_results.append({
            "file": filename,
            "characters": len(text),
            "NFC": unicodedata.is_normalized(
                "NFC",
                text
            ),
            "UTF8_readable": True,
            "status": "OK"
        })

    except UnicodeDecodeError as e:

        verification_results.append({
            "file": filename,
            "characters": None,
            "NFC": None,
            "UTF8_readable": False,
            "status": f"UTF-8 ERROR: {e}"
        })

    except Exception as e:

        verification_results.append({
            "file": filename,
            "characters": None,
            "NFC": None,
            "UTF8_readable": False,
            "status": f"ERROR: {e}"
        })


verification_table = pd.DataFrame(
    verification_results
)


display(
    verification_table
)


# Final checks

if not verification_table.empty:

    failed_verification = verification_table[
        verification_table["status"] != "OK"
    ]

    non_nfc = verification_table[
        verification_table["NFC"] != True
    ]

    print()
    print("Verification summary")

    print(
        f"Text files checked: "
        f"{len(verification_table):,}"
    )

    print(
        f"Verification failures: "
        f"{len(failed_verification):,}"
    )

    print(
        f"Files not in NFC: "
        f"{len(non_nfc):,}"
    )

    if not failed_verification.empty:

        print()
        print("Verification failures:")

        display(
            failed_verification
        )

    if not non_nfc.empty:

        print()
        print("Files not in NFC:")

        display(
            non_nfc
        )

else:

    print()
    print("No text files were produced.")


print()
print("Finished.")


PDF extraction report


,file,pages,pages_with_text,characters,non_whitespace,images,status,reason
0,arz.pdf,6,6,8082,8076,0,SUSPICIOUS,Contains 1 unexpected control characters
1,chn.pdf,7,7,5921,5914,0,TEXT OK,No obvious Unicode extraction problems detected
2,eng.pdf,8,8,11046,11039,0,TEXT OK,No obvious Unicode extraction problems detected
3,hbr.pdf,4,4,11499,11498,0,SUSPICIOUS,"Contains 5,648 unexpected control characters"
4,hnd.pdf,10,1,49,48,8,TEXT OK,No obvious Unicode extraction problems detected
5,jpn.pdf,9,9,4757,4749,0,TEXT OK,No obvious Unicode extraction problems detected
6,por.pdf,6,6,11679,11674,0,TEXT OK,No obvious Unicode extraction problems detected
7,quy.pdf,10,10,13402,13392,0,TEXT OK,No obvious Unicode extraction problems detected
8,rus.pdf,10,10,12206,12200,0,TEXT OK,No obvious Unicode extraction problems detected
9,spn.pdf,9,9,12403,12396,0,TEXT OK,No obvious Unicode extraction problems detected



PDFs requiring inspection


,file,pages,pages_with_text,characters,non_whitespace,images,status,reason
0,arz.pdf,6,6,8082,8076,0,SUSPICIOUS,Contains 1 unexpected control characters
3,hbr.pdf,4,4,11499,11498,0,SUSPICIOUS,"Contains 5,648 unexpected control characters"



PDFs with apparently usable text


...


PDFs passing extraction check: 8
PDFs not automatically converted: 2

Text previews

chn.pdf
' \n世界人权宣言 \n联合国大会一九四八年十二月十日第217A(III)号决议通过并颁布 \n1948 年 12 月 10 日， 联 合 国 大 会 通 过 并 颁 布《 世 界 人 权 宣 言》。 这 一 \n具 有 历 史 意 义 的《 宣 言》 颁 布 后， 大 会 要 求 所 有 会 员 国 广 为 宣 传， \n并 且“ 不 分 国 家 或 领 土 的 政 治 地 位 , 主 要 在 各 级 学 校 和 其 他 教 育 机 \n构 加 以 传 播、 展 示、 阅 读 和 阐 述。” 《 宣 言 》 全 文 如 下： \n序 言 \n鉴 于 对 人 类 家 庭 所 有 成 员 的 固 有 尊 严 及 其 平 等 的 和 不 移 的 权 利 的 \n承 认, 乃 是 世 界 自 由、 正 义 与 和 平 的 基 础,  \n鉴 于 对 人 权 的 无 视 和 侮 蔑 已 发 展 为 野 蛮 暴 行, 这 些 暴 行 玷 污 了 人 类 \n的 良 心, 而 一 个 人 人 享 有 言 论 和 信 仰 自 由 并 免 予 恐 惧 和 匮 乏 的 世 界 \n的 来 临, 已 被 宣 布 为 普 通 人 '

eng.pdf
' \nUniversal Declaration of Human Rights \nPreamble \nWhereas recognition of the inherent dignity and of the equal and inalienable \nrights of all members of the human family is the foundation of freedom, justice \nand peace in the world,  \nWhereas disregard and contempt for human rights have resulted in barbarous \nacts which have outraged the conscience of mankind, and the ad

,file,txt_file,characters,NFC,status
0,chn.pdf,chn.txt,5921,True,SAVED
1,eng.pdf,eng.txt,11046,True,SAVED
2,hnd.pdf,hnd.txt,49,True,SAVED
3,jpn.pdf,jpn.txt,4757,True,SAVED
4,por.pdf,por.txt,11679,True,SAVED
5,quy.pdf,quy.txt,13402,True,SAVED
6,rus.pdf,rus.txt,12206,True,SAVED
7,spn.pdf,spn.txt,12403,True,SAVED



Verifying saved text files


,file,characters,NFC,UTF8_readable,status
0,chn.txt,5921,True,True,OK
1,eng.txt,11046,True,True,OK
2,hnd.txt,49,True,True,OK
3,jpn.txt,4757,True,True,OK
4,por.txt,11679,True,True,OK
5,quy.txt,13402,True,True,OK
6,rus.txt,12206,True,True,OK
7,spn.txt,12403,True,True,OK



Verification summary
Text files checked: 8
Verification failures: 0
Files not in NFC: 0

Finished.


In [ ]:
# # Functions for PDF inspection

# def extract_pdf_text(pdf_path):

#     doc = fitz.open(pdf_path)

#     page_texts = []
#     total_images = 0
#     pages_with_text = 0

#     for page in doc:

#         text = page.get_text()

#         page_texts.append(text)

#         if text.strip():
#             pages_with_text += 1

#         total_images += len(
#             page.get_images(full=True)
#         )

#     doc.close()

#     text = "".join(page_texts)

#     return {
#         "text": text,
#         "pages": len(page_texts),
#         "pages_with_text": pages_with_text,
#         "images": total_images
#     }


# def inspect_text_quality(text):
#     """
#     Look for signs that extracted PDF text is not usable Unicode.

#     This does NOT attempt to determine whether the language itself
#     is linguistically correct. It looks for obvious extraction failures.
#     """

#     stripped = text.strip()

#     if not stripped:
#         return {
#             "quality": "NO TEXT",
#             "reason": "No text extracted"
#         }

#     replacement_count = text.count("\ufffd")

#     control_count = sum(
#         1
#         for char in text
#         if unicodedata.category(char) == "Cc"
#         and char not in "\n\r\t"
#     )

#     unknown_count = sum(
#         1
#         for char in text
#         if unicodedata.name(
#             char,
#             "UNKNOWN"
#         ) == "UNKNOWN"
#     )

#     # Characters with very unusual Unicode categories
#     private_use_count = sum(
#         1
#         for char in text
#         if unicodedata.category(char) == "Co"
#     )

#     if replacement_count > 0:

#         return {
#             "quality": "SUSPICIOUS",
#             "reason": (
#                 f"Contains {replacement_count:,} "
#                 "Unicode replacement characters"
#             )
#         }

#     if private_use_count > 0:

#         return {
#             "quality": "SUSPICIOUS",
#             "reason": (
#                 f"Contains {private_use_count:,} "
#                 "private-use characters"
#             )
#         }

#     if control_count > 0:

#         return {
#             "quality": "SUSPICIOUS",
#             "reason": (
#                 f"Contains {control_count:,} "
#                 "unexpected control characters"
#             )
#         }

#     return {
#         "quality": "TEXT OK",
#         "reason": "No obvious Unicode extraction problems detected"
#     }


# # Inspect all PDFs

# pdf_results = []

# for filename in sorted(os.listdir(pdf_dir)):

#     if not filename.lower().endswith(".pdf"):
#         continue

#     pdf_path = os.path.join(
#         pdf_dir,
#         filename
#     )

#     try:

#         result = extract_pdf_text(
#             pdf_path
#         )

#         text = result["text"]

#         quality = inspect_text_quality(
#             text
#         )

#         pdf_results.append({
#             "file": filename,
#             "pages": result["pages"],
#             "pages_with_text": result["pages_with_text"],
#             "characters": len(text),
#             "non_whitespace": len(text.strip()),
#             "images": result["images"],
#             "status": quality["quality"],
#             "reason": quality["reason"]
#         })

#     except Exception as e:

#         pdf_results.append({
#             "file": filename,
#             "pages": None,
#             "pages_with_text": None,
#             "characters": None,
#             "non_whitespace": None,
#             "images": None,
#             "status": "ERROR",
#             "reason": str(e)
#         })


# pdf_summary = pd.DataFrame(
#     pdf_results
# )


# # Display PDF extraction report

# print()
# print("PDF extraction report")

# display(
#     pdf_summary
# )


# # Display problematic PDFs

# problem_statuses = [
#     "NO TEXT",
#     "SUSPICIOUS",
#     "ERROR"
# ]

# problems = pdf_summary[
#     pdf_summary["status"].isin(
#         problem_statuses
#     )
# ]

# print()
# print("PDFs requiring inspection")

# if len(problems) > 0:
#     display(problems)
# else:
#     print("None")


# # Display PDFs with extracted text

# print()
# print("PDFs with apparently usable text")

# display(
#     pdf_summary[
#         pdf_summary["status"] == "TEXT OK"
#     ]
# )


# # Show text previews for every PDF

# print()
# print("Text previews")

# for filename in sorted(
#     pdf_summary["file"]
# ):

#     pdf_path = os.path.join(
#         pdf_dir,
#         filename
#     )

#     if not os.path.exists(pdf_path):
#         continue

#     result = extract_pdf_text(
#         pdf_path
#     )

#     print()
#     print("=" * 60)
#     print(filename)
#     print("=" * 60)

#     print(
#         repr(
#             result["text"][:500]
#         )
#     )


# # Save only PDFs that pass the basic extraction check

# successful_files = pdf_summary[
#     pdf_summary["status"] == "TEXT OK"
# ]["file"].tolist()


# print()
# print(
#     f"PDFs passing extraction check: "
#     f"{len(successful_files)}"
# )

# print(
#     f"PDFs not automatically converted: "
#     f"{len(pdf_summary) - len(successful_files)}"
# )


# # Convert successful PDFs to UTF-8 text

# conversion_results = []

# for filename in successful_files:

#     pdf_path = os.path.join(
#         pdf_dir,
#         filename
#     )

#     txt_filename = (
#         os.path.splitext(filename)[0]
#         + ".txt"
#     )

#     txt_path = os.path.join(
#         txt_output_dir,
#         txt_filename
#     )

#     try:

#         result = extract_pdf_text(
#             pdf_path
#         )

#         text = result["text"]

#         # NFC normalization

#         text = unicodedata.normalize(
#             "NFC",
#             text
#         )

#         # Verify UTF-8 can represent the text

#         text.encode("utf-8")

#         # Verify NFC

#         is_nfc = unicodedata.is_normalized(
#             "NFC",
#             text
#         )

#         with open(
#             txt_path,
#             "w",
#             encoding="utf-8"
#         ) as f:

#             f.write(text)

#         conversion_results.append({
#             "file": filename,
#             "txt_file": txt_filename,
#             "characters": len(text),
#             "NFC": is_nfc,
#             "status": "SAVED"
#         })

#         print(
#             f"{filename} → {txt_filename} "
#             f"({len(text):,} characters)"
#         )

#     except Exception as e:

#         conversion_results.append({
#             "file": filename,
#             "txt_file": txt_filename,
#             "characters": None,
#             "NFC": None,
#             "status": f"ERROR: {e}"
#         })

#         print(
#             f"{filename}: ERROR"
#         )
#         print(e)


# conversion_table = pd.DataFrame(
#     conversion_results
# )


# # Display conversion results

# print()
# print("Text conversion results")

# display(
#     conversion_table
# )


# # Verify resulting UTF-8 text files

# print()
# print("Verifying saved text files")

# verification_results = []

# for filename in sorted(
#     os.listdir(txt_output_dir)
# ):

#     if not filename.endswith(".txt"):
#         continue

#     path = os.path.join(
#         txt_output_dir,
#         filename
#     )

#     try:

#         with open(
#             path,
#             "r",
#             encoding="utf-8"
#         ) as f:

#             text = f.read()

#         verification_results.append({
#             "file": filename,
#             "characters": len(text),
#             "NFC": unicodedata.is_normalized(
#                 "NFC",
#                 text
#             ),
#             "UTF8_readable": True,
#             "status": "OK"
#         })

#     except UnicodeDecodeError as e:

#         verification_results.append({
#             "file": filename,
#             "characters": None,
#             "NFC": None,
#             "UTF8_readable": False,
#             "status": f"UTF-8 ERROR: {e}"
#         })

#     except Exception as e:

#         verification_results.append({
#             "file": filename,
#             "characters": None,
#             "NFC": None,
#             "UTF8_readable": False,
#             "status": f"ERROR: {e}"
#         })


# verification_table = pd.DataFrame(
#     verification_results
# )

# display(
#     verification_table
# )


# print()
# print("Finished.")

ModuleNotFoundError: No module named 'fitz'